In [11]:
from pathlib import Path

tum_parcalar = []

for dosya in Path("../data").glob("*.md"):
    icerik = dosya.read_text(encoding="utf-8")

    ana_baslik = ""
    bolum_baslik = ""

    for blok in icerik.split("\n\n"):
        blok = blok.strip()

        if not blok:
            continue

        if blok.startswith("## "):
            bolum_baslik = blok.lstrip("# ")
            continue

        if blok.startswith("# "):
            ana_baslik = blok.lstrip("# ")
            bolum_baslik = ""
            continue

        if len(blok) < 10:
            continue

        metin = f"{ana_baslik} — {bolum_baslik}\n\n{blok}"

        tum_parcalar.append({
            "metin": metin,
            "kaynak": dosya.name,
            "bolum": bolum_baslik
        })

print(f"Toplam {len(tum_parcalar)} parça\n")

for p in tum_parcalar[:3]:
    print(f"[{p['kaynak']}] ({len(p['metin'])} karakter)")
    print(p['metin'][:200])
    print("-" * 60)

uzunluklar = [len(p['metin']) for p in tum_parcalar]
print(f"En kısa: {min(uzunluklar)}, En uzun: {max(uzunluklar)}")

from datetime import date

BUGUN = date.today().strftime("%d.%m.%Y")

DOGUM_YILI = 2005
DOGUM_AYI = 11

def yas_hesapla():
    bugun = date.today()
    yas = bugun.year - DOGUM_YILI
    if bugun.month < DOGUM_AYI:
        yas -= 1
    return yas

Toplam 36 parça

[egitim.md] (188 karakter)
Eğitim — Üniversite

Gazi Üniversitesi Elektrik-Elektronik Mühendisliği bölümünde 3. sınıf öğrencisi. Ankara'da yaşıyor ve okuyor. Lisans eğitimi devam ediyor, tahmini mezuniyet 2028 yılı.
------------------------------------------------------------
[egitim.md] (390 karakter)
Eğitim — Bölüm değişikliği

Üniversiteye 2023 yılında diş hekimliği bölümünde başladı. 2024 yılında Gazi Üniversitesi Elektrik-Elektronik Mühendisliği bölümüne geçiş yaptı. Geçiş sebebi, mühendisliğin
------------------------------------------------------------
[egitim.md] (81 karakter)
Eğitim — Lise

Yavuz Sultan Selim Anadolu Lisesi mezunu, 2023 yılında mezun oldu.
------------------------------------------------------------
En kısa: 35, En uzun: 474


In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

metinler = [p["metin"] for p in tum_parcalar]
vektorler = model.encode(metinler, show_progress_bar=True)

print(vektorler.shape)

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.12it/s]

(36, 384)


In [13]:
import torch

def ara(soru, k=3):
    soru_vektoru = model.encode([soru])
    skorlar = model.similarity(soru_vektoru, vektorler)[0]
    degerler, siralar = torch.topk(skorlar, k=k)

    print(f"SORU: {soru}\n")
    for skor, i in zip(degerler, siralar):
        p = tum_parcalar[i]
        print(f"{skor:.3f}  [{p['kaynak']} / {p['bolum']}]")
        print(f"       {p['metin'][:180]}...")
        print()

In [14]:
ara("Hangi üniversitede okuyor?")
ara("Gömülü sistem deneyimi var mı?")
ara("Hangi yarışmalara katıldı?")
ara("PyTorch biliyor mu?")
ara("En sevdiği film ne?")

SORU: Hangi üniversitede okuyor?

0.361  [egitim.md / Üniversite]
       Eğitim — Üniversite

Gazi Üniversitesi Elektrik-Elektronik Mühendisliği bölümünde 3. sınıf öğrencisi. Ankara'da yaşıyor ve okuyor. Lisans eğitimi devam ediyor, tahmini mezuniyet 20...

0.334  [egitim.md / Yabancı dil]
       Eğitim — Yabancı dil

İngilizce seviyesi B2. Teknik dokümantasyon okuma, makale takip etme ve İngilizce kaynaklarla çalışma konusunda rahat....

0.320  [egitim.md / Bölüm değişikliği]
       Eğitim — Bölüm değişikliği

Üniversiteye 2023 yılında diş hekimliği bölümünde başladı. 2024 yılında Gazi Üniversitesi Elektrik-Elektronik Mühendisliği bölümüne geçiş yaptı. Geçiş s...

SORU: Gömülü sistem deneyimi var mı?

0.457  [yetkinlikler.md / Büyük dil modelleri ve RAG]
       Teknik Yetkinlikler — Büyük dil modelleri ve RAG

Çok ajanlı (multi-agent) sistem tasarımı, prompt mühendisliği ve RAG mimarisi üzerine çalıştı. Embedding modelleri, vektör benzerl...

0.456  [yetkinlikler.md / Donanım ve devre

In [15]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("Anahtar yüklendi:", os.getenv("GOOGLE_API_KEY") is not None)

Anahtar yüklendi: True


In [16]:
SISTEM_PROMPT = """Sen Ahmed Çakıroğlu'nun kişisel sitesindeki asistansın.
Ziyaretçiler Ahmed hakkında soru soruyor, sen sadece aşağıda verilen bilgilere
dayanarak cevap veriyorsun.

Kurallar:
- Sadece BİLGİLER bölümündeki içeriği kullan. Kendi genel bilgini kullanma.
- Bilgilerde cevap yoksa "Bu konuda bilgim yok" de. Tahmin yürütme, uydurma.
- Kısa ve net cevap ver, 2-3 cümleyi geçme.
- Türkçe cevap ver, samimi ama profesyonel bir dille.
- Sana bugünün tarihi veriliyor. Doğum tarihi gibi bilgilerden yaş
  hesaplaman gerekirse bu tarihi kullanabilirsin, bu bir istisnadir."""


def cevapla(soru, k=3):
    soru_vektoru = model.encode([soru])
    skorlar = model.similarity(soru_vektoru, vektorler)[0]
    degerler, siralar = torch.topk(skorlar, k=k)

    baglam = "\n\n---\n\n".join(tum_parcalar[i]["metin"] for i in siralar)

    mesaj = f"""BİLGİLER:

{baglam}

SORU: {soru}"""
    sistem = f"{SISTEM_PROMPT}\n\nBugünün tarihi: {BUGUN}. Ahmed şu an {yas_hesapla()} yaşında."

    yanit = client.interactions.create(
        model="gemini-3.5-flash-lite",
        system_instruction=sistem,
        input=mesaj,
        generation_config={
            "temperature": 0.2,
            "max_output_tokens": 500,
            "thinking_level": "minimal"
        }
    )

    return yanit.output_text

In [17]:
sorular = [
    "Hangi üniversitede okuyor?",
    "Gömülü sistem deneyimi var mı?",
    "Hangi yarışmalara katıldı?",
    "En sevdiği film ne?",
    "Kaç yaşında?"
]

for s in sorular:
    print(f"S: {s}")
    print(f"C: {cevapla(s)}")
    print("-" * 70)

S: Hangi üniversitede okuyor?
C: Ahmed, Gazi Üniversitesi Elektrik-Elektronik Mühendisliği bölümünde okumaktadır. Ankara'da yaşamaktadır ve eğitimine devam etmektedir.
----------------------------------------------------------------------
S: Gömülü sistem deneyimi var mı?
C: Evet, gömülü sistemler ve mikrodenetleyici programlama alanında ders düzeyinde bilgi sahibidir ve bu alanı ilerletmek istemektedir. Ayrıca elektrik-elektronik mühendisliği müfredatı kapsamında analog devre tasarımı, LTSpice simülasyonu ve PCB tasarımı konularında ders ve proje deneyimi bulunmaktadır.
----------------------------------------------------------------------
S: Hangi yarışmalara katıldı?
C: Ahmed, TEKNOFEST 2025 kapsamında düzenelenen Robotaksi Otonom Araç Yarışması'na katılmıştır. HÜMA Rover takımıyla bu yarışmada finalist olmuştur.
----------------------------------------------------------------------
S: En sevdiği film ne?
C: Bu konuda bilgim yok.
-----------------------------------------------------

In [ ]:
test_seti = [
    # egitim.md
    {"soru": "Hangi üniversitede okuyor?", "kaynak": "egitim.md", "bolumler": ["Üniversite"]},
    {"soru": "Bölümü ne?", "kaynak": "egitim.md", "bolumler": ["Üniversite"]},
    {"soru": "Neden diş hekimliğini bıraktı?", "kaynak": "egitim.md", "bolumler": ["Bölüm değişikliği"]},
    {"soru": "Hangi liseden mezun?", "kaynak": "egitim.md", "bolumler": ["Lise"]},
    {"soru": "İngilizcesi nasıl?", "kaynak": "egitim.md", "bolumler": ["Yabancı dil"]},
    {"soru": "Hangi sertifikaları var?", "kaynak": "egitim.md", "bolumler": ["Sertifikalar ve kurslar"]},

    # projeler.md
    {"soru": "Otonom araç projesinde ne yaptı?", "kaynak": "projeler.md",
     "bolumler": ["TEKNOFEST 2025 — Robotaksi Otonom Araç Yarışması"]},
    {"soru": "Twitter botu nasıl çalışıyor?", "kaynak": "projeler.md",
     "bolumler": ["Yapay zeka destekli X (Twitter) haber botu"]},
    {"soru": "Tıbbi görüntü üzerine çalışma yaptı mı?", "kaynak": "projeler.md",
     "bolumler": ["Göğüs röntgeni sınıflandırma (CNN)"]},
    {"soru": "Güç kaynağı tasarladı mı?", "kaynak": "projeler.md",
     "bolumler": ["EEE212 — DC güç kaynağı tasarımı"]},
    {"soru": "Kendi web sitesi var mı?", "kaynak": "projeler.md",
     "bolumler": ["Kişisel portfolyo web sitesi"]},

    # yetkinlikler.md
    {"soru": "Hangi programlama dillerini biliyor?", "kaynak": "yetkinlikler.md",
     "bolumler": ["Programlama dilleri"]},
    {"soru": "PyTorch kullanıyor mu?", "kaynak": "yetkinlikler.md",
     "bolumler": ["Makine öğrenmesi ve derin öğrenme"]},
    {"soru": "ROS deneyimi var mı?", "kaynak": "yetkinlikler.md", "bolumler": ["Robotik"]},
    {"soru": "Hangi araçları kullanıyor?", "kaynak": "yetkinlikler.md", "bolumler": ["Araçlar"]},
    {"soru": "PCB tasarımı yapabiliyor mu?", "kaynak": "yetkinlikler.md",
     "bolumler": ["Donanım ve devre tasarımı"]},

    # kariyer.md
    {"soru": "Nerede çalışmak istiyor?", "kaynak": "kariyer.md", "bolumler": ["Hedeflenen sektör"]},
    {"soru": "Yüksek lisans yapacak mı?", "kaynak": "kariyer.md", "bolumler": ["Yüksek lisans planı"]},
    {"soru": "Edge AI nedir, neden ilgileniyor?", "kaynak": "kariyer.md",
     "bolumler": ["Edge AI ve gömülü yapay zeka"]},

    # kisisel.md
    {"soru": "Kaç yaşında?", "kaynak": "kisisel.md", "bolumler": ["Yaşı"]},
    {"soru": "Spor yapıyor mu?", "kaynak": "kisisel.md", "bolumler": ["Spor"]},
    {"soru": "Ona nasıl ulaşabilirim?", "kaynak": "kisisel.md", "bolumler": ["İletişim"]},

    # cevabı olmayanlar
    {"soru": "En sevdiği yemek ne?", "kaynak": None, "bolumler": []},
    {"soru": "Hangi arabayı kullanıyor?", "kaynak": None, "bolumler": []},
    {"soru": "Kardeşi var mı?", "kaynak": None, "bolumler": []},
    {"soru": "Evcil hayvanı var mı?", "kaynak": None, "bolumler": []},
]

print(f"{len(test_seti)} soru")

26 soru


In [19]:
def degerlendir(k_degerleri=(1, 3, 5)):
    max_k = max(k_degerleri)
    cevapli = [t for t in test_seti if t["kaynak"] is not None]
    cevapsiz = [t for t in test_seti if t["kaynak"] is None]

    isabet = {k: 0 for k in k_degerleri}
    kacanlar = []

    for t in cevapli:
        sv = model.encode([t["soru"]])
        skorlar = model.similarity(sv, vektorler)[0]
        _, siralar = torch.topk(skorlar, k=max_k)

        bulunan_sira = None
        for sira, i in enumerate(siralar):
            p = tum_parcalar[i]
            if p["kaynak"] == t["kaynak"] and p["bolum"] in t["bolumler"]:
                bulunan_sira = sira
                break

        for k in k_degerleri:
            if bulunan_sira is not None and bulunan_sira < k:
                isabet[k] += 1

        if bulunan_sira is None or bulunan_sira >= 3:
            en_iyi = tum_parcalar[siralar[0]]
            kacanlar.append((t["soru"], f"{en_iyi['kaynak']} / {en_iyi['bolum']}"))

    print(f"Cevaplanabilir soru sayısı: {len(cevapli)}\n")
    for k in k_degerleri:
        oran = isabet[k] / len(cevapli)
        print(f"Recall@{k}: {isabet[k]}/{len(cevapli)} = {oran:.1%}")

    print(f"\nİlk 3'te bulunamayanlar ({len(kacanlar)}):")
    for soru, geldi in kacanlar:
        print(f"  '{soru}'  →  {geldi}")

    print("\nCevabı olmayan sorularda en yüksek skorlar:")
    for t in cevapsiz:
        sv = model.encode([t["soru"]])
        skorlar = model.similarity(sv, vektorler)[0]
        print(f"  {skorlar.max():.3f}  '{t['soru']}'")


degerlendir()

Cevaplanabilir soru sayısı: 22

Recall@1: 13/22 = 59.1%
Recall@3: 18/22 = 81.8%
Recall@5: 19/22 = 86.4%

İlk 3'te bulunamayanlar (4):
  'Bölümü ne?'  →  projeler.md / AhmedGPT — RAG tabanlı kişisel chatbot
  'Tıbbi görüntü üzerine çalışma yaptı mı?'  →  yetkinlikler.md / Robotik
  'Hangi araçları kullanıyor?'  →  yetkinlikler.md / Büyük dil modelleri ve RAG
  'Yüksek lisans yapacak mı?'  →  egitim.md / Üniversite

Cevabı olmayan sorularda en yüksek skorlar:
  0.185  'En sevdiği yemek ne?'
  0.201  'Hangi arabayı kullanıyor?'
  0.206  'Kardeşi var mı?'
  0.184  'Evcil hayvanı var mı?'


In [20]:
dogru_skorlar = []

for t in test_seti:
    if t["kaynak"] is None:
        continue
    sv = model.encode([t["soru"]])
    skorlar = model.similarity(sv, vektorler)[0]
    _, siralar = torch.topk(skorlar, k=5)
    for i in siralar:
        p = tum_parcalar[i]
        if p["kaynak"] == t["kaynak"] and p["bolum"] in t["bolumler"]:
            dogru_skorlar.append((skorlar[i].item(), t["soru"]))
            break

dogru_skorlar.sort()
print("Doğru parçanın skorları (en düşükten):")
for s, soru in dogru_skorlar[:6]:
    print(f"  {s:.3f}  '{soru}'")
print(f"\nEn düşük doğru skor: {dogru_skorlar[0][0]:.3f}")
print(f"En yüksek cevapsız skor: 0.206")

Doğru parçanın skorları (en düşükten):
  0.126  'Ona nasıl ulaşabilirim?'
  0.259  'Kaç yaşında?'
  0.264  'Hangi sertifikaları var?'
  0.266  'Tıbbi görüntü üzerine çalışma yaptı mı?'
  0.303  'Nerede çalışmak istiyor?'
  0.345  'Neden diş hekimliğini bıraktı?'

En düşük doğru skor: 0.126
En yüksek cevapsız skor: 0.206
